# Black–Litterman portfolio optimization with macro forecasts

This notebook tests whether return forecasts from macroeconomic factors improve portfolio allocation. A Ridge model turns information available at each historical decision date into Black–Litterman views. Equal weight, Markowitz, BL Momentum, and buy-and-hold SPY provide comparisons.

The main experiment uses historical ALFRED snapshots to avoid training on later revisions. It requires a free FRED API key. A separate `price_only` mode is available for checking the price pipeline, but it does not answer the research question.

Select **Kernel → Restart Kernel and Run All Cells** after setting the key. Calculations are in `src/`; results depend on the downloaded data and fixed parameters.


In [ ]:
from pathlib import Path
import os
import json
import hashlib
import importlib.metadata
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Start Jupyter from the repository root.
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the repository root.')

from src.data import load_prices, monthly_schedule, load_macro_features
from src.backtest import run_backtest
from src.plots import plot_prices, plot_results, plot_weights, plot_risk_return

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.float_format = '{:,.4f}'.format

## 1. A small, diversified universe

The ETFs existed before the sample begins: US equities (SPY), developed equities outside the US (EFA), emerging markets (EEM), US Treasury bonds (IEF), and gold (GLD). All are listed in USD.

This choice reduces the temptation to select today's winning individual stocks, but the universe was still chosen retrospectively. SPY is an equity reference; equal weight is a closer comparison within the same multi-asset universe.


In [ ]:
ASSETS = ['SPY', 'EFA', 'EEM', 'IEF', 'GLD']
BENCHMARK = 'SPY'
START = '2010-01-01'
END = '2026-01-15'  # Fixed date; January closes the final holding period.
CACHE = ROOT / 'data' / 'cache'
OUTPUT = ROOT / 'results'

# The main experiment requires vintage macro data. Use 'price_only' only for diagnostics.
RUN_MODE = os.environ.get('RUN_MODE', 'macro')  # 'price_only' for diagnostics
FRED_API_KEY = os.environ.get('FRED_API_KEY', '')
if RUN_MODE not in {'macro', 'price_only'}:
    raise ValueError("RUN_MODE must be 'macro' or 'price_only'.")
if RUN_MODE == 'macro' and not FRED_API_KEY:
    raise RuntimeError('The macro experiment requires FRED_API_KEY. See the README for setup.')

PARAMETERS = dict(lookback=36, ridge_window=60, min_train=24,
                  validation_months=12, rf_annual=0.02, risk_aversion=3.0,
                  tau=0.05, momentum_strength=0.005, ridge_alpha=1.0, cost_bps=10.0)

display(pd.Series(PARAMETERS, name='Parameters fixed before the test'))
print('Run mode:', RUN_MODE)


In [ ]:
prices = load_prices(list(dict.fromkeys(ASSETS + [BENCHMARK])), START, END, CACHE)
display(prices.head())
display(pd.DataFrame({'First date': [prices.index.min()], 'Last date': [prices.index.max()],
                      'Trading sessions': [len(prices)], 'Missing values': [int(prices.isna().sum().sum())]}))
plot_prices(prices)
plt.show()

## 2. Chronology matters as much as the formula

1. Signal at the month's final trading close: estimates use only prices available through that date.
2. Buy at the first close of the following month, leaving one session between signal and execution.
3. Hold until the next month's first close, then rebalance.

A Ridge target enters training only after its holding period has ended **before the current signal**. The still-open period is never used for training. An incomplete final month creates no artificial monthly return.


In [ ]:
schedule = monthly_schedule(prices)
display(schedule.head(6))

## 3. Macro forecasts using information available at the time

The main run requests ALFRED snapshots through the FRED API. For each signal, `realtime_start = realtime_end = the day before the signal`, so later data revisions cannot replace what was known then.

The factors are the change in the 10-year Treasury yield (`GS10`), the change in VIX (`VIXCLS`), CPI inflation (`CPIAUCSL`), and industrial production growth (`INDPRO`). Each uses the latest two completed observation months available for that series. Publication lags differ.

The initial download can take several minutes (four requests per signal date), and subsequent runs reuse the cache. A FRED API key is required for uncached historical snapshots. The explicit `price_only` mode skips BL Ridge and cannot answer the macro research question.


In [ ]:
macro = None
if RUN_MODE == 'macro':
    macro = load_macro_features(schedule.signal, FRED_API_KEY, CACHE)
    display(macro.tail())
    macro.plot(subplots=True, figsize=(11, 8), title='Factors available at decision dates', legend=True)
    plt.tight_layout()
    plt.show()
else:
    print('Price-only diagnostic: BL Ridge is excluded, so this run cannot answer the macro question.')


## 4. From the prior to portfolio weights

All calculations use **monthly** returns. The monthly risk-free rate is $r_f=(1+r_f^{annual})^{1/12}-1$.

Covariance is estimated from 36 monthly returns and shrunk 20% toward its diagonal. The excess-return prior is $\pi=\delta\Sigma w_0$, with $w_0=(1/N,\ldots,1/N)$ and $\delta=3$. This is a **reference-allocation prior**, not a market portfolio reconstructed from today's capitalizations.

Momentum: $m_i=p_{i,t-1}/p_{i,t-12}-1$, then $Q_i=\pi_i+0.005\,z_i$. The latest month is excluded, and $z_i$ standardizes the signal across assets.

Ridge: factors known at the signal predict the next holding period's return. $Q_i=\widehat R_i-r_f$. Uncertainty is the MSE of 12 chronological out-of-sample predictions whose targets are already known. The model and scaling are then refitted using up to 60 available observations.

With absolute views ($P=I$), Black–Litterman combines $\pi$ and $Q$. Weights then maximize $w^T\mu-\frac{\delta}{2}w^T\Sigma w$ subject to $\sum w_i=1$ and $w_i\geq0$. See the README and `src/model.py` for details.


In [ ]:
result = run_backtest(prices, ASSETS, benchmark=BENCHMARK, macro=macro, **PARAMETERS)
if RUN_MODE == 'macro' and 'BL Ridge' not in result['returns']:
    raise RuntimeError('The macro run did not produce BL Ridge results.')
print(f"Common period: {result['wealth'].index[0].date()} → {result['wealth'].index[-1].date()}")
print(f"{len(result['returns'])} monthly periods; strategies: {', '.join(result['returns'].columns)}")
display(result['timeline'].head())


## 5. Results: return, risk, and costs

Wealth curves start at 100 before the first transaction cost. Buying 100% of capital costs 10 basis points; selling one asset and buying another incurs costs on both amounts. SPY pays only its entry cost. Positions are not liquidated at the end of the test.

Drawdown is measured from the previous peak **observed monthly**; it does not capture lows within a month. A higher return is less compelling if risk or turnover rises substantially.


In [ ]:
percent_columns = ['Total return', 'CAGR', 'Volatility', 'Max drawdown', 'Annual turnover']
formats = {column: '{:.1%}' for column in percent_columns}
formats['Sharpe'] = '{:.2f}'
display(result['metrics'].style.format(formats))
plot_results(result)
plt.show()
plot_risk_return(result)
plt.show()

## 6. What does the model buy?

The stacked area chart shows target weights at each trade. Sharp changes indicate sensitivity to forecasts; persistent concentration deserves attention even when performance appears strong.


In [ ]:
if 'BL Ridge' in result['returns']:
    plot_weights(result, 'BL Ridge')
    plt.show()
plot_weights(result, 'BL Momentum')
plt.show()
display(result['weights'].groupby(level='strategy').tail(1))

## 7. Check sensitivity to transaction costs

These scenarios change only costs (0, 10, or 25 basis points), not parameters selected to maximize performance. They show whether turnover consumes an apparent advantage.


In [ ]:
sensitivity = {}
for bps in [0, 10, 25]:
    scenario = result if bps == PARAMETERS['cost_bps'] else run_backtest(
        prices, ASSETS, benchmark=BENCHMARK, macro=macro, **{**PARAMETERS, 'cost_bps': bps})
    sensitivity[f'{bps} bps'] = scenario['metrics']['CAGR']
display(pd.DataFrame(sensitivity).style.format('{:.1%}'))

## 8. Save and interpret the results

Tables, charts, and parameters are exported to `results/`. The manifest records the run mode, versions, and fingerprints of the data actually used. Preserve `data/cache/` to reproduce the run; these local files are not versioned.

For the main experiment, compare BL Ridge with BL Momentum, equal weight, and Markowitz. Discuss risk, concentration, and costs. Do not infer predictive ability from a single period. A poor macro result or no improvement is a valid finding. The `price_only` mode is only a pipeline diagnostic.


In [ ]:
OUTPUT.mkdir(exist_ok=True)
for name in ['returns', 'metrics', 'weights', 'turnover', 'fees', 'timeline']:
    result[name].to_csv(OUTPUT / f'{name}.csv')
for name, figure in [('performance', plot_results(result)),
                     ('allocation', plot_weights(result, 'BL Ridge' if macro is not None else 'BL Momentum')),
                     ('risk_return', plot_risk_return(result))]:
    figure.savefig(OUTPUT / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.close(figure)
manifest = {
    'assets': ASSETS, 'benchmark': BENCHMARK, 'start': START, 'end': END,
    'parameters': PARAMETERS, 'run_mode': RUN_MODE, 'macro_alfred': macro is not None,
    'python': sys.version,
    'packages': {name: importlib.metadata.version(name) for name in
                 ['numpy', 'pandas', 'pandas_market_calendars', 'scipy', 'matplotlib', 'yfinance', 'requests']},
    'prices_sha256': hashlib.sha256(prices.to_csv().encode()).hexdigest(),
    'macro_sha256': hashlib.sha256(macro.to_csv().encode()).hexdigest() if macro is not None else None,
    'first_execution': str(result['wealth'].index[0].date()),
    'last_exit': str(result['wealth'].index[-1].date()),
}
(OUTPUT / 'run.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Exports:', OUTPUT.relative_to(ROOT))


### Limitations to retain in the conclusion

The universe was selected retrospectively; adjusted Yahoo prices are downloaded today; taxes, variable spreads, market impact, and the actual dividend schedule are omitted. Transaction costs are approximate, the risk-free rate is constant, parameters are not calibrated, and risk is observed only monthly. ALFRED snapshots address later macro revisions but do not remove every limitation of a historical experiment.

This is an educational experiment, not investment advice.
